# 自定义属性与主题

学习目标：能复用颜色和间距，追踪自定义属性的继承与回退，并实现系统偏好和显式选择共同控制的主题。

前置知识：CSS 声明、层叠与继承、选择器、颜色和尺寸、媒体查询及 HTML 属性。

适用范围：CSS Custom Properties Level 1 与 CSS Properties and Values API Level 1 的 @property；后者对应工作草案中的注册机制，旧实现须保留普通自定义属性或固定值回退。 配套页面不依赖 JavaScript。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/16-custom-properties/。

1. [index.html](scripts/16-custom-properties/index.html)：复用、继承与作用范围。
2. [invalid.html](scripts/16-custom-properties/invalid.html)：回退与计算值无效。
3. [registered.html](scripts/16-custom-properties/registered.html)：注册属性与普通值回退。
4. [themes.html](scripts/16-custom-properties/themes.html)：跟随系统主题。
5. [light.html](scripts/16-custom-properties/light.html)：固定浅色主题。
6. [dark.html](scripts/16-custom-properties/dark.html)：固定暗色主题。
7. [styles.css](scripts/16-custom-properties/styles.css)：本章各页面的实验规则及少量阅读辅助样式。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/16-custom-properties/index.html)。

服务根目录为 content/Web与应用开发/css；保存修改后刷新页面

Step 4：在服务终端按 Ctrl+C 停止服务。

本章使用的主要属性：

| 完整属性名 | 中文名称／含义 | 用途或作用对象 |
| --- | --- | --- |
| --brand | 本例品牌颜色 | 作者定义的自定义属性 |
| --space | 本例基础间距 | 作者定义的自定义属性 |
| --surface | 本例主题表面颜色 | 作者定义的自定义属性 |
| --text | 本例主题文字颜色 | 作者定义的自定义属性 |
| --link | 本例主题链接颜色 | 作者定义的自定义属性 |
| --badge-tone | 本例注册的颜色属性 | 限制为颜色，默认不继承 |
| color | 前景颜色 | 接收颜色值 |
| background-color | 背景颜色 | 接收颜色值 |
| padding | 内边距简写 | 复用间距 |
| gap | 布局间距 | 复用间距 |
| color-scheme | 元素接受的颜色方案 | 影响原生控件、画布和滚动条等浏览器绘制 |
| border-color | 边框颜色 | 复用主题边界颜色 |

## 1 用自定义属性复用颜色与间距

自定义属性（custom property）以两个连字符开头，由作者命名；--brand 是本例名称，没有内置的“品牌”效果。只有被 color、padding 等属性使用后，才影响这些呈现。

var(--brand) 读取当前元素的 --brand 值，var() 是值中的替换函数。名称区分大小写，--brand 与 --Brand 是两个属性。:root 是选中根元素的伪类，常用于放置共享值；并非所有自定义属性都必须定义在根上。

值保留合适的类型和单位。--space: 12px 可以直接用于 padding，也可通过 calc(var(--space) * 2) 得到双倍间距；不能靠 var(--number)px 把数字和 px 拼成一个长度。var() 不是预处理器，不能拿来拼属性名或选择器。

```html
<div class="token-list">
  <article class="token-card"><h3>颜色复用</h3><p>边框与文字使用同一颜色。</p></article>
  <article class="token-card"><h3>间距复用</h3><p>卡片与列表使用同一基础间距。</p></article>
</div>
```

```css
/* 基础令牌在根元素定义，卡片复用颜色与间距。 */
:root {
  --brand: #174f79;
  --space: 12px;
}
.token-list { display: grid; gap: var(--space); }
.token-card {
  color: var(--brand);
  border: 2px solid var(--brand);
  padding: var(--space);
}
.token-card h3 { margin-top: 0; }
/* 临时修改根元素的 --brand 或 --space，检查两张卡片及 gap 同时变化。 */
```

配套文件：[index.html](scripts/16-custom-properties/index.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/index.html)

## 2 继承、局部覆盖与作用范围

未注册的普通自定义属性默认继承，也遵守层叠。祖先上的值可以传给后代；后代声明自己的值会覆盖该处继承值，不会改写祖先或兄弟元素。

“作用范围”在这里来自选择器匹配与继承，不是 JavaScript 变量的词法作用域。局部组件直接设 --brand，即可复用同一份使用规则；若无处定义某个名称，不能从兄弟组件借值。

本例先修改局部祖先的属性，再检查其两层后代。不要只看自定义属性声明存在，还要查看实际消费它的 color 是否被其他规则覆盖。

```html
<div class="scope-default"><p class="scope-text">根颜色</p></div>
<div class="scope-local"><p class="scope-text">局部颜色 <span>后代继续继承</span></p></div>
<p class="scope-text scope-sibling">局部组件以外的兄弟</p>
```

```css
.scope-local { --brand: #754300; }
.scope-text { color: var(--brand); padding: 8px; border: 1px solid currentColor; }
/* 局部段落及 span 使用棕色；外面的兄弟仍使用根颜色。 */
```

配套文件：[index.html](scripts/16-custom-properties/index.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/index.html)

## 3 var() 的缺失回退与循环依赖

var() 第一个参数是自定义属性名，逗号之后是该引用不可用时的回退。回退也可以包含 var()；第一个逗号之后的全部内容属于一个回退值，不是依次尝试多个普通值。

对未注册属性，未定义、设置为 initial，或出现同一元素上的循环依赖等情况会产生保证无效值（guaranteed-invalid value），可以触发回退。回退仍必须适合最后使用它的属性。

--a 引用 --b，而 --b 又引用 --a，会形成循环；给 var(--a, teal) 加回退只能在消费处提供颜色，不能修复循环本身。这里故意展示循环，实际主题应使用无环的值依赖。

var() 回退也不是给完全不支持自定义属性的浏览器使用：那类实现会忽略含 var() 的声明，需要前置普通声明。

```html
<p class="missing-fallback">缺少主颜色时采用后备颜色</p>
<p class="cycle-fallback">循环导致引用无效时采用回退</p>
<p class="literal-fallback">保留普通声明作为语法不支持回退</p>
```

```css
.missing-fallback {
  --backup: teal;
  color: var(--missing, var(--backup, black));
}
.cycle-fallback {
  --a: var(--b);
  --b: var(--a);
  color: var(--a, teal);
  /* 反例：两个名称形成循环，消费处应采用 teal。 */
}
.literal-fallback {
  color: teal;
  color: var(--missing, teal);
}
```

配套文件：[invalid.html](scripts/16-custom-properties/invalid.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/invalid.html)

## 4 类型错误不一定使用 var() 回退

--wrong: 16px 对未注册自定义属性本身是合法值；但把它代入 color 会形成无效颜色。由于 --wrong 已有可替换值，var(--wrong, red) 不会先做 color 类型检查再选 red。

带 var() 的声明先参与层叠，替换后才发现类型不适合，这称为计算值阶段无效（invalid at computed-value time）。此时不会回头使用被它覆盖的前一条颜色声明。

普通继承属性 color 在这种情况下取继承值；非继承属性 background-color 使用初始值 transparent。直接写 color: 16px 则能较早被忽略，保留前面的有效声明。这两类失败不能混为一谈。

```html
<div class="invalid-parent">
  <p class="literal-invalid">字面颜色错误</p>
  <p class="computed-invalid">替换后的类型错误</p>
  <p class="background-invalid">背景替换后的类型错误</p>
</div>
```

```css
.invalid-parent { color: #174f79; background-color: #f2f2f2; padding: 8px; }
.literal-invalid {
  color: teal;
  color: 16px; /* 有意的反例：无效字面值被忽略，仍是 teal。 */
}
.computed-invalid {
  --wrong: 16px;
  color: teal;
  color: var(--wrong, red);
  /* 反例：不是 red，也不退回 teal；color 继承父元素的深蓝色。 */
}
.background-invalid {
  --wrong: 16px;
  background-color: gold;
  background-color: var(--wrong, red);
  /* 反例：计算背景为 transparent，透出父背景，不代表自身背景被继承。 */
}
```

配套文件：[invalid.html](scripts/16-custom-properties/invalid.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/invalid.html)

## 5 依赖值在定义处计算，再继承

自定义属性中的 var() 在该元素上计算后再被继承。祖先定义的别名不会因为后代改了另一个名称，就自动在后代重新求值。

下面父元素把 --alias 解析为 teal；子元素只修改 --source 为棕色，但继承下来的 --alias 已经是 teal。若希望跟随子元素的新来源，可在子元素重新声明 --alias: var(--source)，或直接消费 --source。

这解释了多层主题中常见的“局部变量改了，别名仍没变”；排查时分别检查源属性、别名计算值和最终颜色。

```html
<div class="alias-parent">
  <p class="alias-inherited">继承已经计算的别名</p>
  <p class="alias-rebound">在本元素重新计算别名</p>
</div>
```

```css
.alias-parent { --source: teal; --alias: var(--source); }
.alias-inherited, .alias-rebound { --source: #754300; color: var(--alias); }
.alias-rebound {
  --alias: var(--source);
  /* 第一段仍为 teal，第二段根据本地 --source 变为棕色。 */
}
```

配套文件：[index.html](scripts/16-custom-properties/index.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/index.html)

## 6 @property 声明类型、初始值与继承行为

@property 是注册自定义属性的 @ 规则，不能写成普通选择器声明。注册描述的是这个名称的行为；它不把注册限制在某个类的后代，普通选择器仍负责给元素赋值。

规则中的三个名称是描述符，不放入属性总览表：

- syntax：允许的类型字符串，本例 "&lt;color&gt;" 表示颜色，星号 "*" 表示通用语法。
- inherits：是否默认继承，本例 false 表示未在自身赋值时使用注册初始值；显式写 inherit 仍能要求继承。
- initial-value：注册初始值；除了通用语法 "*" 的情况，通常必须提供，且要能独立计算。固定颜色或 10px 可以，依赖元素字号的 3em 或 var() 不适合这样的初始值。

syntax 和 inherits 必需；缺少必需描述符会使整个注册规则无效。类型检查发生在计算值阶段，不是让类型错误重新回到更早声明。注册也为后续动画提供按类型插值的基础，本节只观察静态值。

本例 inherits: false：普通子元素没有自身赋值时用 teal，错误的 16px 也落到注册初始值；var() 中的 orange 不会替代已存在的注册初始值。旧浏览器忽略 @property 后会按未注册属性处理，颜色可能不同，正文仍可阅读；下方显式值回退组不依赖注册。

```html
<div class="registered-parent">
  <p class="registered-child">未在自身赋值</p>
  <p class="registered-invalid">自身给出错误类型</p>
  <p class="registered-inherit">显式要求继承</p>
</div>
<div class="plain-parent"><p class="plain-fallback">普通自定义属性的显式值回退</p></div>
```

```css
/* registered：注册类型和继承方式，与未注册属性作对照。 */
@property --badge-tone {
  syntax: "<color>";
  inherits: false;
  initial-value: teal;
}
.registered-parent { --badge-tone: #754300; }
.registered-child, .registered-invalid, .registered-inherit {
  color: var(--badge-tone, orange);
  border: 1px solid currentColor;
  padding: 8px;
}
.registered-invalid { --badge-tone: 16px; }
.registered-inherit { --badge-tone: inherit; }
.plain-parent { --plain-tone: #754300; }
.plain-fallback {
  --plain-tone: teal;
  color: teal;
  color: var(--plain-tone, teal);
  /* 支持注册时前三段为 teal、teal、棕色；本回退段明确为 teal。 */
}
```

配套文件：[registered.html](scripts/16-custom-properties/registered.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/registered.html)

## 7 主题值与系统明暗偏好

主题可以把表面、文字、链接与边框颜色集中在语义名称里，再由组件消费。切换时必须一起考虑前景和背景，不能只把页面背景涂黑；链接与焦点也需要可辨认。

prefers-color-scheme 是 @media 的媒体特性，dark 表示偏好暗色，light 也涵盖没有主动表达暗色偏好的情况。媒体条件只决定何时应用声明，不会自动为作者颜色生成另一套主题。

本例以 data-theme="auto" 明确表示“跟随系统”。data-theme 是本例 HTML 自定义数据属性约定；[data-theme="auto"] 是属性选择器，浏览器没有预设的主题切换含义。其他示例页不带这个标记，避免被主题实验干扰。

```html
<html lang="zh-CN" data-theme="auto">
```

```html
<article class="theme-panel">
  <h3>阅读主题</h3>
  <p>同一组件读取表面、文字、链接和边框值。</p>
  <a class="theme-link" href="#theme-end">阅读下一段</a>
  <p><label>笔记标题 <input class="theme-input" value="布局学习"></label></p>
</article>
<p id="theme-end">下一段内容。</p>
```

```css
/* 主题先给出浅色令牌，下面分别处理系统偏好和固定选择。 */
:root {
  --surface: #ffffff;
  --text: #17202a;
  --link: #174f79;
  --edge: #677789;
}
:root[data-theme] body {
  background-color: var(--surface);
  color: var(--text);
}
:root[data-theme] a { color: var(--link); }
.theme-panel {
  padding: var(--space);
  border: 2px solid var(--edge);
  background-color: var(--surface);
  color: var(--text);
}
.theme-input { max-width: 100%; }
/* 只让 auto 跟随系统；显式 light / dark 由后面的规则确定。 */
@media (prefers-color-scheme: dark) {
  :root[data-theme="auto"] {
    --surface: #17202a;
    --text: #f4f7fa;
    --link: #a8d5ff;
    --edge: #b6c5d4;
    /* 仿真 dark：作者前景、背景、链接和边框都切换到对应值。 */
  }
}
```

配套文件：[themes.html](scripts/16-custom-properties/themes.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/themes.html)

## 8 显式选择主题并设置 color-scheme

显式选择应有明确的覆盖规则。本例把 light、dark 作为 data-theme 的另外两个值；系统查询只匹配 auto，因此明确选择不受系统偏好改变影响。页面链接用于切换三份独立可运行的 HTML，不需要 JavaScript，也不保存用户设置。

color-scheme 是 CSS 属性，告诉浏览器元素可接受的明暗方案，影响原生输入框、滚动条、画布等默认绘制。light dark 允许根据偏好选择，单独 light 或 dark 则约束此处方案。它不会把作者写死的所有颜色自动重新配色。

只有组件确实准备了两套可读样式，才声明接受两种方案。手动暗色主题也要同时设 color-scheme: dark，否则作者背景与原生控件可能不一致；不支持该属性时仍保留作者颜色和可操作的原生控件。

本例不覆盖输入框的前景与背景，让它展示原生方案。原生控件具体外观依赖浏览器和系统；还应在强制颜色、缩放和键盘状态下检查，而不是以颜色值匹配代替可读性评估。

```html
<p class="theme-choices">
  <a href="themes.html">跟随系统</a>
  <a href="light.html">固定浅色</a>
  <a href="dark.html">固定暗色</a>
</p>
```

```css
:root[data-theme="auto"] { color-scheme: light dark; }
:root[data-theme="light"] { color-scheme: light; }
:root[data-theme="dark"] {
  color-scheme: dark;
  --surface: #17202a;
  --text: #f4f7fa;
  --link: #a8d5ff;
  --edge: #b6c5d4;
}
.theme-choices { display: flex; flex-wrap: wrap; gap: 16px; }
.theme-link:focus-visible, .theme-input:focus-visible {
  outline: 3px solid var(--link);
  outline-offset: 3px;
}
/* 点击固定浅色后再仿真系统 dark：作者主题和 color-scheme 仍保持 light。 */
```

配套文件：[themes.html](scripts/16-custom-properties/themes.html)、[styles.css](scripts/16-custom-properties/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/16-custom-properties/themes.html)

## 本章小结

- 自定义属性通过 var() 被实际属性消费，名称区分大小写，并参与层叠与继承。
- var() 回退处理不可用的引用，不为任意类型错误兜底；计算值阶段无效不会重新寻找更早声明。
- 祖先别名计算后再继承，局部源值变化不保证让继承别名重新计算。
- @property 的 syntax、inherits、initial-value 是描述符；注册初始值和 var() 回退作用不同。
- 主题同时配置作者颜色和 color-scheme，明确选择与系统偏好的关系，并保留原生交互。

## 练习

（1）把首页根 --space 改为 20px，检查卡片内边距和列表间隔；只在第二张卡片上改 --brand，检查第一张不变。

（2）在 invalid.html 中先把 --wrong 改为有效颜色，再改回 16px。比较两个阶段的最终 color 与 background-color，解释为什么 red 回退没有处理错误类型。

（3）把注册的 inherits 改为 true，预测未在自身赋值和显式 inherit 两段的结果；在计算样式中核对，不把旧结果照搬。

（4）分别打开固定浅色、固定暗色页，并设置相反系统偏好。检查作者颜色、color-scheme 和键盘焦点一致，表单仍可输入。

### 提示

修改注册描述符要刷新源文件后的页面；不要只看 var() 的文本。主题的配色需要连同控件、链接、禁用或焦点状态一起评估。

## 参考与引用来源

- W3C：[CSS Custom Properties Level 1 §2](https://www.w3.org/TR/css-variables-1/#defining-variables)、[循环依赖](https://www.w3.org/TR/css-variables-1/#cycles)、[§3](https://www.w3.org/TR/css-variables-1/#using-variables)：名称、继承、替换、回退与计算值无效；[CSS Properties and Values API Level 1 §2](https://www.w3.org/TR/css-properties-values-api-1/#behavior-of-custom-properties)、[@property](https://www.w3.org/TR/css-properties-values-api-1/#at-property-rule)：类型、继承、初始值和注册行为；[CSS Color Adjustment 的 color-scheme](https://www.w3.org/TR/css-color-adjust-1/#color-scheme-prop)：浏览器默认配色与作者颜色的分工。
- MDN：[Using custom properties](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Cascading_variables/Using_custom_properties)、[var()](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/var)、[CSS error handling](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Syntax/Error_handling)、[Property value processing](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Cascade/Property_value_processing)：值复用、嵌套回退、计算值错误和继承计算值；[@property](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@property)、[Registering properties](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Properties_and_values_API/Registering_properties)：描述符条件与支持；[prefers-color-scheme](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@media/prefers-color-scheme)、[color-scheme](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/color-scheme)：偏好媒体特性、原生控件与主题颜色。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface)：服务工作目录、端口和绑定地址。